# Fuel Burn Calculations

The mass of the fuel consumed throughout the flight, and possibly consumed during the reserve mission as well, is calculated in aviary_group.py.
The fuel mass can be estimated in premission by assuming that the difference between the gross mass and the zero fuel mass is given to fuel.
However, for off-design missions, this estimate will be inaccurate. For many design missions it can also be inaccurate.
These issues can be compounded by the fact that the aircraft could have some payload it was dropping during the mission. 
Deployable payloads (payloads that are not delivered to the final destination) currently do not have any place in the overall aircraft mass buildup.

![Fuel Breakdown](images/fuel_breakdown.png "Details on the fuel breakdown is given in the figure.")


## Fuel Mass
The Fuel Mass breakdown is shown below. Note that UNUSABLE_FUEL_MASS is not accounted for in this mass and is instead book kept under USEFUL_LOAD which contributes to OPERATING_MASS see [mass subsystem for details](mass.ipynb).

- TOTAL_FUEL_MASS (mass)
    - FUEL_BURNED (mass)
    - RESERVE_FUEL (mass)
        - RESERVE_FUEL_BURNED (mass)
        - RESERVE_FUEL_ADDITIONAL (mass)
        - RESERVE_FUEL_MARGIN (%)

## Maximum Fuel Mass 
The total mass fuel that can fit into the aircraft, regardless of fuel mass or gross mass constraints is as follows:
- TOTAL_CAPACITY (mass)
    - WING_FUEL_CAPACITY (mass)
    - FUSEALGE_FUEL_CAPACITY (mass)
    - AUXILIARY_FUEL_CAPACITY (mass)
    - UNUSABLE_FUEL_MASS (mass)

## Constraints
A fuel mass constraint is used to ensure the fuel mass does not exceed the total fuel allowed in the fuel tanks.
This is essentially a volume constraint but we track it as a mass because all the fuel-tank capacities are tracked as mass not volume.
This is enforced by evaluating {glue:md}`Mission.Constraints.EXCESS_FUEL_CAPACITY`. 
If this value is below zero, then there is more fuel mass than capacity in the tanks. 
This constraint can be disabled by setting {glue:md}`Aircraft.Fuel.IGNORE_FUEL_CAPACITY_CONSTRAINT` to True.

A gross mass constraint is used to ensure fuel mass never pushes the aircraft over the design gross mass of the vehicle.
This constraint is alaways enforced by {glue:md}`Mission.Constraints.MASS_RESIDUAL`.
This is especially relevant in alternate missions where the aircraft has been designed and you are trying to assess how much fuel is needed to complete an off-design mission.
If we let the fuel push the aircraft over the design gross mass, the landing gear can fail.
During design missions, this sort of violation will force the optimizer to choose a larger value for design gross mass which will cause resizing in all the other supporting systems.

## Conversions from GASP to Aviary Fuel
Caution should be taken when converting GASP aircraft and evaluating fuel mass.
FRESF, in GASP can be assigned to three differen values: an extra fuel weight in lbs, an extra fuel percentage, or an extra reserve flight time.
FRESF values between 0 and 10 will not evaluate to a extra reserve flight time. Instead this must be accomplished by setting up a reserve mission and a `target_time`. Doing this will help Aviary calculate {glue:md}`Mission.Summary.RESERVE_FUEL_BURNED` which is the fuel burn for all of the reserve missions. [Read more about generating reserve missions.](../reserve_missions.ipynb)
FRESF values at or below 0 will convert to a percentage of mission fuel burn reserive and bookept under {glue:md}`Aircraft.Design.RESERVE_FUEL_MARGIN`. (`FRESHF = -.1` -> `RESERVE_FUEL_MARGIN = 10`)
FRESF values at or above 10 will convert directly to lbm fuel burn reserves and accounted for in {glue:md}`Aircraft.Design.RESERVE_FUEL_ADDITIONAL`. 
Further details on these conversiosn can be viewed in `fortran_to_aviary.py`.

In [ ]:
# Testing Cell
from aviary.api import Aircraft, Mission
from aviary.utils.doctape import get_variable_name, glue_variable

glue_variable(get_variable_name(Mission.Constraints.EXCESS_FUEL_CAPACITY), md_code=True)
glue_variable(get_variable_name(Aircraft.Fuel.IGNORE_FUEL_CAPACITY_CONSTRAINT), md_code=True)
glue_variable(get_variable_name(Mission.Constraints.MASS_RESIDUAL), md_code=True)
glue_variable(get_variable_name(Aircraft.Design.RESERVE_FUEL_MARGIN), md_code=True)
glue_variable(get_variable_name(Aircraft.Design.RESERVE_FUEL_ADDITIONAL), md_code=True)
glue_variable(get_variable_name(Mission.Summary.RESERVE_FUEL_BURNED), md_code=True)